In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
 
TARGET_TABLE = "main.bronze.silver_customers"  # <-- change this
BRONZE_TABLE = "main.bronze.bronze_customers"   # <-- adjust if different
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### Clean slate
# MAGIC Drop the test table so re-running this notebook always starts fresh.
 
# COMMAND ----------
 

df = spark.read.table(BRONZE_TABLE)
from pyspark.sql import functions as F

cleaned_data = (
    df.select(
        F.col("customer_id"),
        F.trim(F.col("name")).alias("customer_name"),
        F.lower(F.trim(F.col("email"))).alias("email"),
        F.trim(F.col("phone")).alias("phone"),
        F.col("loyalty_tier").alias("loyalty_tier"),
        F.col("updated_at"),
    )
    .withColumn(
        "customer_sk",
        F.sha2(
            F.concat_ws("||", F.col("customer_id"), F.col("email"), F.col("phone"), F.col("loyalty_tier")),
            256,
        ),
    )
)

window = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())

latest_rows = (
    cleaned_data
    .withColumn("_rn", F.row_number().over(window))
    .filter("_rn = 1")
    .drop("_rn")
)

current_silver = spark.read.table(TARGET_TABLE).filter("__START_AT IS NOT NULL AND __END_AT IS NULL")

changed_or_new = (
    latest_rows.alias("s")
    .join(current_silver.alias("t"), on="customer_id", how="left")
    .filter(
        "t.customer_id IS NULL "
        "OR NOT (LOWER(TRIM(t.email)) <=> LOWER(TRIM(s.email))) "
        "OR NOT (TRIM(t.customer_name) <=> TRIM(s.customer_name)) "
        "OR NOT (TRIM(t.phone) <=> TRIM(s.phone)) "
        "OR NOT (t.loyalty_tier <=> s.loyalty_tier)"
    )
    .select("s.*")
)

close_out_rows = latest_rows.withColumn("_merge_key", F.col("customer_id").cast("string"))
insert_rows = changed_or_new.withColumn("_merge_key", F.lit(None).cast("string"))

staged = (close_out_rows.unionByName(insert_rows)).filter("customer_id == 1")
staged.show(truncate=False)

spark.read.table(BRONZE_TABLE).filter(F.col("customer_id")  ==1).orderBy(F.col("updated_at").desc()).show(truncate=False)


In [0]:
 
TARGET_TABLE = "main.bronze.silver_customers"  # <-- change this
BRONZE_TABLE = "main.bronze.bronze_customers"   # <-- adjust if different


df_bronze = spark.read.table(BRONZE_TABLE).filter("customer_id = 1").orderBy(F.col("updated_at").desc())
df_bronze.show()

df_silver = spark.read.table(TARGET_TABLE).filter("customer_id = 1").orderBy(F.col("updated_at").desc())
df_silver.show()


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BRONZE_TABLE = "main.bronze.silver_customers"   # <-- adjust if different

df = spark.read.table(BRONZE_TABLE)
df2 = spark.read.table("main.gold.dim_customers")

joined = df.alias("d1").join(df2.alias("d2"), on="customer_id", how="left")
            
display("COUNT ==",joined.count())